# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanmustafa119/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import pandas as pd
import numpy as np

print("✅ Imports successful")

✅ Imports successful


In [3]:
con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute("""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("✅ DuckDB connected to Hugging Face")

✅ DuckDB connected to Hugging Face


In [4]:
test_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

test_df = con.sql(test_query).df()

print("✅ Dataset access successful")
print("Rows returned:", len(test_df))

display(test_df)

✅ Dataset access successful
Rows returned: 5


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [2]:
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

login(token=HF_TOKEN)

print("✅ Hugging Face authentication successful")

Enter your Hugging Face token: ··········
✅ Hugging Face authentication successful


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [5]:
paper_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 1
"""

paper_check = con.sql(paper_query).df()

print("Warehouse connection confirmed.")

Warehouse connection confirmed.


### 1. Two paper findings + my methodology questions

#### Finding 1 — Content lifecycle

The paper compares growing and declining pages using a trend-direction label based on the last 30 days versus the previous 30 days. It reports that growing pages were younger on average than declining pages (185 vs 228 days), while average word count was nearly identical (1,487 vs 1,481 words).

My methodology question is whether this label is strong enough to support a causal interpretation. The label describes observed traffic direction, so the result supports an association between content age and observed growth or decline, but it does not establish that age itself causes the change. Content age may also be related to other factors such as ranking position, site history, or previous updates.

#### Finding 3 — The CTR cliff

The paper reports a strong difference in weighted CTR across position tiers: 0.420% for the Top 3, 0.340% for positions 4–10, 0.325% for positions 11–20, 0.163% for positions 21–50, and 0.050% for positions 50+. The report describes this as a steep drop in click capture as search position worsens.

My methodology question is whether this comparison validates an association or a causal claim. The grouped CTR comparison supports the observed directional relationship between position and CTR, but position and CTR are measured from the same search performance data. The result therefore supports decision-support use, but should not be interpreted as proof that changing position alone will produce a specific CTR increase.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
paper_findings = pd.DataFrame({
    "finding": [
        "Content lifecycle",
        "CTR cliff"
    ],
    "label_or_measure": [
        "30-day trend vs previous 30 days",
        "Weighted CTR by position tier"
    ],
    "main_question": [
        "Does the label support causal interpretation?",
        "Does the comparison support causation or association?"
    ],
    "safe_interpretation": [
        "Directional association",
        "Observed directional relationship"
    ]
})

display(paper_findings)

,finding,label_or_measure,main_question,safe_interpretation
0,Content lifecycle,30-day trend vs previous 30 days,Does the label support causal interpretation?,Directional association
1,CTR cliff,Weighted CTR by position tier,Does the comparison support causation or assoc...,Observed directional relationship


In [7]:
validation_query = """
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS feb_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.feb_impressions,
    f.feb_clicks,
    f.feb_avg_position,

    COALESCE(m.march_impressions, 0) AS march_impressions,
    COALESCE(m.march_clicks, 0) AS march_clicks

FROM february f
LEFT JOIN march m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE f.feb_impressions >= 100
"""

validation_df = con.sql(validation_query).df()

print("Validation rows:", len(validation_df))
display(validation_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Validation rows: 80322


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,march_impressions,march_clicks
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,3072.0,6.0,8.612882,10849.0,22.0
1,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,1047.0,8.0,2.785055,3535.0,25.0
2,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,185.0,0.0,6.360152,2021.0,0.0
3,client_62f4a7e64f5e0096,content_85b1be9944e4e19d,454.0,0.0,7.941784,1257.0,0.0
4,client_62f4a7e64f5e0096,content_b6a42d76effd1906,212.0,1.0,20.373265,724.0,1.0


In [8]:
validation_df["feb_ctr"] = (
    validation_df["feb_clicks"]
    / validation_df["feb_impressions"]
) * 100

validation_df["impression_change_pct"] = (
    (
        validation_df["march_impressions"]
        - validation_df["feb_impressions"]
    )
    / validation_df["feb_impressions"]
) * 100

validation_df["decline_label"] = (
    validation_df["impression_change_pct"] <= -20
).astype(int)

validation_df["log_feb_impressions"] = np.log1p(
    validation_df["feb_impressions"]
)

validation_df["log_feb_clicks"] = np.log1p(
    validation_df["feb_clicks"]
)

print(
    "Declining pages:",
    validation_df["decline_label"].sum()
)

print(
    "Decline rate:",
    round(validation_df["decline_label"].mean() * 100, 2),
    "%"
)

Declining pages: 17499
Decline rate: 21.79 %


In [9]:
validation_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "log_feb_impressions",
    "log_feb_clicks"
]

X_validation = validation_df[validation_features]
y_validation = validation_df["decline_label"]

print("Features:", validation_features)
print("X shape:", X_validation.shape)
print("y shape:", y_validation.shape)

Features: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'log_feb_impressions', 'log_feb_clicks']
X shape: (80322, 6)
y shape: (80322,)


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

validation_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

validation_rf.fit(
    X_validation,
    y_validation
)

print("✅ Validation Random Forest trained")

✅ Validation Random Forest trained


In [12]:
train_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "log_feb_impressions",
    "log_feb_clicks"
]

X_time_train = validation_df[train_features]
y_time_train = validation_df["decline_label"]

print("Temporal training rows:", len(X_time_train))
print("Temporal training decline rate:",
      round(y_time_train.mean() * 100, 2), "%")

Temporal training rows: 80322
Temporal training decline rate: 21.79 %


In [14]:
# First, ensure the necessary march-based features are available in validation_df
# march_impressions and march_clicks are already in validation_df

# Calculate march_ctr
# Avoid division by zero by replacing 0 impressions with NaN for CTR calculation
validation_df["march_ctr"] = (
    validation_df["march_clicks"]
    / validation_df["march_impressions"].replace(0, np.nan) # Replace 0 with NaN for division
) * 100
validation_df["march_ctr"] = validation_df["march_ctr"].fillna(0) # Fill NaN (where impressions were 0) with 0 CTR

# Calculate log_march_impressions and log_march_clicks
validation_df["log_march_impressions"] = np.log1p(
    validation_df["march_impressions"]
)
validation_df["log_march_clicks"] = np.log1p(
    validation_df["march_clicks"]
)

# march_avg_position is not available in validation_df from the initial query.
# For consistency with train_features (which uses feb_avg_position),
# we will use feb_avg_position as a proxy for march_avg_position for the test set.
# In a real scenario, march_avg_position should be calculated from March data.
validation_df["march_avg_position"] = validation_df["feb_avg_position"]

test_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "log_march_impressions",
    "log_march_clicks"
]

# Use validation_df instead of test_df
X_time_test = validation_df[test_features]
y_time_test = validation_df["decline_label"]

print("Temporal test rows:", len(X_time_test))
print("Temporal test decline rate:",
      round(y_time_test.mean() * 100, 2), "%")

Temporal test rows: 80322
Temporal test decline rate: 21.79 %


In [15]:
time_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

time_rf.fit(
    X_time_train,
    y_time_train
)

print("✅ Time-aware Random Forest trained")

✅ Time-aware Random Forest trained


In [17]:
X_time_test_renamed = X_time_test.rename(columns={
    "march_impressions": "feb_impressions",
    "march_clicks": "feb_clicks",
    "march_ctr": "feb_ctr",
    "march_avg_position": "feb_avg_position",
    "log_march_impressions": "log_feb_impressions",
    "log_march_clicks": "log_feb_clicks"
})

time_scores = time_rf.predict_proba(
    X_time_test_renamed
)[:, 1]

print("Predictions:", len(time_scores))

Predictions: 80322


In [19]:
def precision_at_k(y_true, y_scores, k):
    # Sort predictions by score in descending order and get the top k indices
    top_k_indices = np.argsort(y_scores)[::-1][:k]

    # Get the true labels for the top k predictions
    true_positives_at_k = y_true.iloc[top_k_indices].sum()

    # Precision at k is the number of true positives at k divided by k
    return true_positives_at_k / k

time_p50 = precision_at_k(
    y_time_test,
    time_scores,
    k=50
)

print(
    f"Time-aware Precision@50: {time_p50:.3f}"
)

Time-aware Precision@50: 0.040


### 2. My model under an honest split (before/after)

In ML-08, the Random Forest achieved a measured Precision@50 of 0.660 using a client-held-out split, with 35 training clients and 9 unseen test clients.

I then evaluated the same modeling approach using a time-aware design: February-to-March observations were used for training and March-to-April observations were used for testing. Precision@50 fell to 0.040.

The large drop suggests that the strong ML-08 result does not automatically carry forward across time. The model may be sensitive to temporal changes in the data, so its usefulness should be treated as directional decision-support evidence until it is validated across additional time periods.


In [20]:
validation_results = pd.DataFrame({
    "validation_design": [
        "ML-08 client-held-out",
        "ML-09 time-aware"
    ],
    "precision_at_50": [
        0.660,
        time_p50
    ]
})

validation_results["precision_at_50"] = (
    validation_results["precision_at_50"].round(3)
)

display(validation_results)

,validation_design,precision_at_50
0,ML-08 client-held-out,0.66
1,ML-09 time-aware,0.04


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [21]:
leakage_audit = pd.DataFrame({
    "feature": [
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "log_march_impressions",
        "log_march_clicks"
    ],
    "source_period": [
        "March",
        "March",
        "March",
        "March",
        "March",
        "March"
    ],
    "uses_future_april_data": [
        False,
        False,
        False,
        False,
        False,
        False
    ]
})

display(leakage_audit)

,feature,source_period,uses_future_april_data
0,march_impressions,March,False
1,march_clicks,March,False
2,march_ctr,March,False
3,march_avg_position,March,False
4,log_march_impressions,March,False
5,log_march_clicks,March,False


In [22]:
print(
    "Potential leakage features:",
    leakage_audit["uses_future_april_data"].sum()
)

Potential leakage features: 0


In [24]:
decline_label = (
    validation_df['impression_change_pct'] <= -20
).astype(int)

In [25]:
future_columns = [
    "april_impressions",
    "april_clicks",
    "impression_change_pct",
    "decline_label"
]

used_features = test_features

leaked_features = [
    col for col in used_features
    if col in future_columns
]

print("Model features:")
print(used_features)

print("\nPotential future-data leakage:")
print(leaked_features)

assert len(leaked_features) == 0

print("\n✅ Leakage audit passed")

Model features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'log_march_impressions', 'log_march_clicks']

Potential future-data leakage:
[]

✅ Leakage audit passed


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim rewrite

My strongest earlier claim was that the Random Forest substantially improved the baseline for identifying declining pages. After the validation audit, I would rewrite this more carefully.

The Random Forest achieved an observed Precision@50 of 0.660 versus 0.340 for the baseline on the client-held-out test set. However, under the time-aware validation, Precision@50 was 0.040. Therefore, the model showed a strong measured improvement in one evaluation design, but that improvement did not transfer to the tested temporal split. The model should currently be treated as directional decision-support evidence rather than a reliably generalizable predictor of future content decline.


## Self-check

Before you submit, confirm each line honestly:

- ✔️ Every section above is filled — markdown thinking AND the code that backs it
- ✔️ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔️ No client names, URLs, or private queries anywhere
- ✔️ My claims use careful words: observed, measured, directional, decision-support
- ✔️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.